## Libraries

In [2]:
import os

In [3]:
run = 'local' #local or cloud
os.environ["APP_ENV"] = run

In [4]:
import sys
sys.path.append('..')

In [5]:
from src.core_grass import*
%matplotlib inline
#%load_ext autoreload
#%autoreload 2

In [6]:
import subprocess
result = subprocess.run(["grass", "--config", "python_path"],
                        check=True, text=True, capture_output=True)
sys.path.append(result.stdout.strip())

# import GRASS Python packages
from collections import defaultdict
import grass.script as gs
import grass.jupyter as gj

# Parameters

In [7]:
#set project variables
Project_Area = 'essex'
projection = '3424' #set to None if you want to use the DEM's original projection
res = '1m'

## Set variables analysis
# source = 'wbd' #wbd or user-defined
# data_scale = 'HUC12' #other options HUC12, HUC10, HUC8 OR the field name of the source data to be split <-- case sensitive
# select_data = '030300020407' #     # Value within the filter_field used for data selection 
# to_headwaters = False #False if only interested in local huc area
# input_shp_name = None #None or must be populated populated with a shapefile polygons if data_scale != HUCXX
# buffer = 2000
# auto_delete = False

In [8]:
## Paths:
parent_dir = pl.Path(os.getcwd()).parent/'data'
grass_data_dir = pl.Path(os.getcwd()).parent/'grassdata'
current_dir = pl.Path(os.getcwd())
outputs_dir =parent_dir/'Rasters'/Project_Area/'Outputs'
vector_dir = parent_dir/'Vectors'/Project_Area
raster_dir = parent_dir/'Rasters'/Project_Area
huc_path = vector_dir/'{}_aoi.geojson'.format(Project_Area)
git_data_repo = current_dir/'data_repository'
# if input_shp_name:
#     input_shp = vector_dir/input_shp_name
directory = outputs_dir

### Start new grass session

In [9]:
# define GRASS data settings (adapt to your needs)
location = Project_Area

In [10]:
# Create a project if it doesn't exist
if os.path.exists(grass_data_dir/location):
    pass
else:
    if len(projection) > 4:
        wkt_text = pyproj.CRS.from_authority("esri", int(projection)).to_wkt()
        with open(vector_dir/'{}_prj.wkt'.format(Project_Area), 'w') as t:
            t.write(wkt_text)
        gs.create_project(path=grass_data_dir,
                      name=location,
                      wkt=vector_dir/'{}_prj.wkt'.format(Project_Area))
    else:
        gs.create_project(path=grass_data_dir,
                  name=location,
                  epsg=projection)

In [11]:
# Start GRASS in default project mapset
session = gj.init(grass_data_dir/location)

#### Review existing files

In [12]:
#list all grass files
grass_maps = list_existing_grass(print_it=True)

Available vector maps:
BloomfieldAve_AOI@PERMANENT
BloomfieldAve_AOI_raw@PERMANENT
EC_EnvJustice_combined_streesor_summary@PERMANENT
GI_pnts@PERMANENT
GI_pnts_raw@PERMANENT
GI_pnts_raw_back@PERMANENT
ParkAve_AOI@PERMANENT
ParkAve_AOI_ces@PERMANENT
ParkAve_AOI_cesd@PERMANENT
ParkAve_AOI_cesdf@PERMANENT
ParkAve_AOI_cesdfe@PERMANENT
ParkAve_AOI_cesdfec@PERMANENT
ParkAve_AOI_cesdfec8f@PERMANENT
ParkAve_AOI_cesdfec8f8r@PERMANENT
ParkAve_AOI_cln@PERMANENT
ParkAve_AOI_exp@PERMANENT
ParkAve_AOI_raw@PERMANENT
ROWLimits_nonParcel@PERMANENT
Roads_ROW@PERMANENT
SOrange_AOI@PERMANENT
bloomfield_ces@PERMANENT
bloomfield_cesd@PERMANENT
bloomfield_cesdf@PERMANENT
bloomfield_cesdfc@PERMANENT
bloomfield_cesdfe@PERMANENT
bloomfield_cesdfec@PERMANENT
bloomfield_cesdfec8f@PERMANENT
bloomfield_cesdfec8f8r@PERMANENT
bloomfield_cln@PERMANENT
bloomfield_exp@PERMANENT
essex_county@PERMANENT
essex_nj_BloomfieldAve_AOI@PERMANENT
essex_nj_ParkAve_AOI@PERMANENT
essex_nj_SOrange_AOI@PERMANENT
inlet_essex_nj@PERMANEN

In [13]:
reg = reg = gs.parse_command('g.region',flags='gp')
cell_size = (float(reg.nsres)+ float(reg.ewres)) /2

In [14]:
threshold = 5000//(cell_size**2) #5000 sf
main_select_data= 'essex_nj'
aoi = 'ParkAve_AOI' #ParkAve_AOI, SOrange_AOI BloomfieldAve_AOI
select_data = main_select_data+'_'+aoi
gi_raw = f'{aoi}_raw'

In [15]:
#standard variables
dem = 'dem_essex_nj'
c_lines = f'v_carve_{select_data}'
accum = 'row_accum'
drain_dir = 'row_drain_dir'
r_basins = f'r_basins_{select_data}'
v_basins = f'v_basins_{select_data}'
sub_basins = f'subbasins_{select_data}'
v_streams = f'stream_{select_data}'
r_streams = f'r_streams_{select_data}'
r_streams_o = f'r_streams_o_{select_data}'
twi = f'r_twi_{select_data}'
v_outlet = f'outlet_{select_data}'
v_inlet = f'inlet_{select_data}'

In [16]:
dem

'dem_essex_nj'

## Prep Work

In [16]:
gs.run_command('r.mask',flags='r')

In [17]:
gs.run_command('g.region',raster = dem,align=dem,zoom=dem)

In [18]:
gs.run_command("v.clip", input="ROWLimits_nonParcel", clip=aoi, output=select_data)

In [19]:
gs.run_command("v.clip", input="roads", clip=aoi, output=f"roads_{select_data}")

In [20]:
gs.run_command('r.mask',vector = select_data)
gs.run_command('g.region',raster = dem,align=dem,zoom=dem)
reg = gs.parse_command('g.region',raster = dem,align=dem,zoom=dem,flags='gp')

In [21]:
gs.run_command('r.watershed',elevation = dem, threshold=threshold,accumulation = accum, drainage=drain_dir, stream = r_streams,flags='a')

In [22]:
import re
import subprocess

# 1. Start the stream extraction process asynchronously
p = gs.start_command('r.stream.extract',elevation=dem,accumulation=accum,threshold=threshold,stream_vector =v_streams, direction=drain_dir, memory = 3000,stderr=subprocess.PIPE)

# 2. Read the output in real-time
while True:
    line = p.stderr.readline()
    if not line and p.poll() is not None:
        break
    if line:
        # Decode bytes to string and print (or parse for percentages)
        print(line)

# 3. Finalize process check
rc = p.returncode
if rc == 0:
    print("Process completed successfully.")
else:
    print(f"Process failed with return code {rc}.")

Loading input raster maps...



Initializing A* search...



A* Search...



Extracting streams...



Thinning stream segments...



Writing output raster maps...



Writing vector map <stream_essex_nj_ParkAve_AOI>...


         be overwritten



Writing attribute data...

Building topology for vector map <stream_essex_nj_ParkAve_AOI@PERMANENT>...

Registering primitives...



Process completed successfully.


In [23]:
gs.run_command("v.db.renamecolumn", map=v_streams, column="stream_type,strm_type")

In [24]:
gs.run_command("v.clean", input=v_streams, output=v_streams+'_cln', tool="rmline")
gs.run_command('v.out.ogr',input=v_streams+'_cln',output=vector_dir/f'flow_{v_streams}.shp',type= 'line',format='ESRI_Shapefile')
#gs.run_command('v.out.ogr',input=v_streams,output=vector_dir/f'flow_{v_streams}.GeoJSON',type= 'line',format='GeoJSON')

In [25]:
gs.run_command("v.extract", input=v_streams, output=v_inlet+'_line',type = 'line',where = "(strm_type = 'start')") 
gs.run_command("v.to.points",input=v_inlet+'_line', output=v_inlet, use='start')

In [26]:
gs.run_command('r.stream.basins',direction =drain_dir,points =v_inlet,basins=r_basins)
gs.run_command('r.to.vect',input = r_basins,output = v_basins, type='area',column='basin_col', flags='s')

In [27]:
gs.run_command('v.out.ogr',input=v_basins,output=vector_dir/f'TDA_less_than_5000_{aoi}.shp',type= 'area',format='ESRI_Shapefile')

In [28]:
aoi

'ParkAve_AOI'

### SLOPE

In [29]:
gs.run_command('r.slope.direction',elevation = dem,direction =drain_dir,dir_type ='auto',slope_measure  ='percent',steps =7, output ='slope_dir',flags='a')

In [30]:
gs.mapcalc("steep_10_pct = if({0} > 0.15, 1, null())".format('slope_dir'))

In [31]:
gs.run_command('r.to.vect',input = 'steep_10_pct',output = f'steep_10_pct_{aoi}', type='area',column='steep_col', flags='s')

In [32]:
gs.run_command('v.out.ogr',input=f'steep_10_pct_{aoi}',output=vector_dir/f'slope_more_than_10_pct_{aoi}.shp',type= 'area',format='ESRI_Shapefile')

## Work for GI Screening

In [53]:
#clean
gs.run_command("v.multi2singlepart", input=gi_raw, output=gi_raw.replace('raw','exp'))
gs.run_command("v.clean", input=gi_raw.replace('raw','exp'),output=gi_raw.replace('raw','cln'), tool='rmdangle,rmbridge,bpol,rmarea',threshold='10,20,0,5',flags='c')

In [54]:
gs.run_command('v.out.ogr',input=gi_raw.replace('raw','cln'),output=vector_dir/f'{aoi}_{gi_raw.replace('raw','cln')}.shp',type= 'area',format='ESRI_Shapefile')

In [55]:
#remove steep areas
gs.run_command("v.overlay", ainput=gi_raw.replace('raw','cln'), binput=f'steep_10_pct_{aoi}', operator="not", output=gi_raw.replace('raw','ces'),flags='t')

In [56]:
gs.run_command('v.out.ogr',input=gi_raw.replace('raw','ces'),output=vector_dir/f'{aoi}_{gi_raw.replace('raw','ces')}.shp',type= 'area',format='ESRI_Shapefile')

In [57]:
#remove less than 5000 sf areas
gs.run_command("v.overlay", ainput=gi_raw.replace('raw','ces'), binput=v_basins, operator="not", output=gi_raw.replace('raw','cesd'),flags='t')

In [58]:
gs.run_command('v.out.ogr',input=gi_raw.replace('raw','cesd'),output=vector_dir/f'{aoi}_{gi_raw.replace('raw','cesd')}.shp',type= 'area',format='ESRI_Shapefile')

In [59]:
#remove false connections
gs.run_command('v.buffer',input=gi_raw.replace('raw','cesd'), output=gi_raw.replace('raw','cesdb'),distance = -2)
gs.run_command('v.buffer',input=gi_raw.replace('raw','cesdb'), output=gi_raw.replace('raw','cesdbb'),distance = 2)

In [60]:
#separate by flow path
gs.run_command('v.buffer',input=v_streams+'_cln',output=v_streams+'_cln'+'_buffer',distance = 0.1)
gs.run_command("v.overlay", ainput=gi_raw.replace('raw','cesdbb'), binput=v_streams+'_cln'+'_buffer', operator="not", output=gi_raw.replace('raw','cesdbbf'),flags='t')

In [61]:
#clean
gs.run_command("v.multi2singlepart", input=gi_raw.replace('raw','cesdbbf'), output=gi_raw.replace('raw','cesdbbfe'))
gs.run_command("v.clean", input=gi_raw.replace('raw','cesdbbfe'),output=gi_raw.replace('raw','cesdbbfec'), tool='rmdangle,rmbridge,bpol,rmarea',threshold='10,20,0,5',flags='c')

In [62]:
gs.run_command('v.out.ogr',input=gi_raw.replace('raw','cesdbbfec'),output=vector_dir/f'{aoi}_{gi_raw.replace('raw','cesdbbfec')}.shp',type= 'area',format='ESRI_Shapefile')

In [63]:
#select down
gs.run_command('v.buffer',input=v_streams+'_cln',output=v_streams+'_cln'+'select_buffer',distance = 8)
gs.run_command('v.buffer',input=f"roads_{select_data}",output=f"roads_{select_data}"+'select_buffer',distance = 8)

In [64]:
gs.run_command("v.select", ainput=gi_raw.replace('raw','cesdbbfec'), binput=v_streams+'_cln'+'select_buffer', output=gi_raw.replace('raw','cesdbbfec8f'), operator="intersects",flags='t')
gs.run_command("v.select", ainput=gi_raw.replace('raw','cesdbbfec8f'), binput=f"roads_{select_data}"+'select_buffer', output=gi_raw.replace('raw','cesdbbfec8f8r'), operator="intersects",flags='tr')

In [65]:
gs.run_command('v.db.addtable',map=gi_raw.replace('raw','cesdbbfec8f8r'))

In [66]:
#get raster info
gs.run_command("v.rast.stats", map=gi_raw.replace('raw','cesdbbfec8f8r'), raster="TCC", column_prefix="TCC_Pct",method ='average',flags='cd')

In [67]:
gs.run_command("v.rast.stats", map=gi_raw.replace('raw','cesdbbfec8f8r'), raster="EC_CST", column_prefix="EC_CST_Rank",method ='average',flags='cd')

In [68]:
gs.run_command("v.rast.stats", map=gi_raw.replace('raw','cesdbbfec8f8r'), raster="Wtr_table_depth_ft", column_prefix="Dp2Wtr_ft",method ='minimum',flags='cd')

In [69]:
gs.run_command('v.out.ogr',input=gi_raw.replace('raw','cesdbbfec8f8r'),output=vector_dir/f'{aoi}_{gi_raw.replace('raw','cesdbbfec8f8r')}.geojson',type= 'area',format='GeoJSON')
#convert to shp
gpd.read_file(vector_dir/f'{aoi}_{gi_raw.replace('raw','cesdbbfec8f8r')}.geojson').to_file(vector_dir/f'{aoi}_{gi_raw.replace('raw','cesdbbfec8f8r')}.shp')

/tmp/ipykernel_47/2058219097.py:3: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gpd.read_file(vector_dir/f'{aoi}_{gi_raw.replace('raw','cesdbbfec8f8r')}.geojson').to_file(vector_dir/f'{aoi}_{gi_raw.replace('raw','cesdbbfec8f8r')}.shp')
/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'TCC_Pct_average' to 'TCC_Pct_av'
  ogr_write(
/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'EC_CST_Rank_average' to 'EC_CST_Ran'
  ogr_write(
/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'Dp2Wtr_ft_minimum' to 'Dp2Wtr_ft_'
  ogr_write(


## WORK FOR GI AREAS Attribution

In [70]:
# Load or create your GeoDataFrame
gdf_cent = gpd.read_file(vector_dir/f'{aoi}_{gi_raw.replace('raw','cesdbbfec8f8r')}.geojson')

# Create a new column with centroid points
gdf_cent["geometry"] = gdf_cent.geometry.representative_point()
gdf_cent["basin_col"] = gdf_cent.index+1

gdf_cent.to_file(vector_dir/'GI_Opps_pnts.shp')

/tmp/ipykernel_47/1204159962.py:8: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_cent.to_file(vector_dir/'GI_Opps_pnts.shp')
/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'TCC_Pct_average' to 'TCC_Pct_av'
  ogr_write(
/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'EC_CST_Rank_average' to 'EC_CST_Ran'
  ogr_write(
/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'Dp2Wtr_ft_minimum' to 'Dp2Wtr_ft_'
  ogr_write(


In [71]:
gs.run_command('v.import', input= vector_dir/'GI_Opps_pnts.shp', output= 'GI_pnts_raw')

In [72]:
gs.run_command('g.copy', vector='GI_pnts_raw,GI_pnts_raw_back')

In [73]:
gs.run_command('r.stream.snap',input = 'GI_pnts_raw', output='GI_pnts',stream_rast = r_streams,radius = 50/cell_size)

In [74]:
#gs.run_command('v.db.connect', map='GI_pnts',table='GI_pnts_raw',database ='$GISDBASE/$LOCATION_NAME/PERMANENT/sqlite/sqlite.db', overwrite =True)
gs.run_command('v.db.addtable', map='GI_pnts')
gs.run_command('v.db.join', map='GI_pnts', column='cat', other_table='GI_pnts_raw_back', other_column='cat')

In [75]:
gs.run_command('r.stream.basins',points = 'GI_pnts', direction=drain_dir,basins=r_basins+'GI')

In [76]:
gs.run_command('r.to.vect',input = r_basins+'GI',output = v_basins+'GI', type='area',column='basin_col', flags='s')

In [77]:
gs.run_command('v.out.ogr',input=v_basins+'GI',output=vector_dir/f'{v_basins}_GI.geojson',type= 'area',format='GeoJSON')
#convert to shp
gpd.read_file(vector_dir/f'{v_basins}_GI.geojson').to_file(vector_dir/f'{v_basins}_GI.shp')

In [78]:
basin_data = gpd.read_file(vector_dir/f'{v_basins}_GI.shp')
basin_data.set_index('basin_col',inplace=True)
#attach shape info
basin_data['TDA_sf'] = basin_data.geometry.area
basin_data['TDA_acre'] = basin_data.geometry.area/43560
tda_ref = basin_data[['TDA_sf']].to_dict()

/tmp/ipykernel_47/1379404839.py:2: Pandas4Warning: The inplace keyword in GeoDataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  basin_data.set_index('basin_col',inplace=True)


In [86]:
GI_data = gpd.read_file(vector_dir/f'{aoi}_{gi_raw.replace('raw','cesdbbfec8f8r')}.shp')
GI_data['basin_col'] = GI_data.index+1
#attach shape info
GI_data['Area_sf'] = GI_data.geometry.area
GI_data['Perim_ft'] = GI_data.geometry.length
GI_data['Width_ft'] = (2*GI_data.geometry.area)/GI_data.geometry.length
GI_data['Length_ft'] = GI_data.geometry.area/GI_data['Width_ft']
GI_data['LR'] = GI_data.apply(lambda x: (tda_ref['TDA_sf'][x.basin_col]/x.Area_sf) if x.basin_col in tda_ref['TDA_sf'].keys() else 0,axis=1)

In [87]:
#lat long
GI_data['Lat'] = GI_data.geometry.centroid.to_crs(4269).y
GI_data['Lon'] = GI_data.geometry.centroid.to_crs(4269).x

In [88]:
#filter out widths less than 3.5 feet.
GI_data_cut = GI_data.loc[GI_data['Width_ft']>=3.5]

In [89]:
#add soils data
GI_data_cut

,cat,TCC_Pct_av,EC_CST_Ran,Dp2Wtr_ft_,geometry,basin_col,Area_sf,Perim_ft,Width_ft,Length_ft,LR,Lat,Lon
0,4,NaN,16.000000,14.996720,"POLYGON ((582674.735 700160.998, 582674.472 70...",1,659.769204,228.150743,5.783625,114.075371,19.045138,40.755138,-74.172943
1,10,NaN,14.000000,23.106956,"POLYGON ((581792.737 700385.065, 581792.472 70...",2,186.794360,53.048152,7.042445,26.524076,49.172240,40.755779,-74.176290
2,22,0.000000,14.000000,18.008531,"POLYGON ((580622.631 700993.51, 580622.449 700...",3,221.175680,86.317857,5.124680,43.158929,0.133904,40.757485,-74.180504
3,27,NaN,18.000000,25.000001,"POLYGON ((579162.716 701538.948, 579162.479 70...",4,546.790188,118.568098,9.223226,59.284049,21.633575,40.759017,-74.185750
4,42,0.000000,15.000000,25.003282,"POLYGON ((577630.427 702032.097, 577630.19 702...",5,2984.482288,434.145385,13.748769,217.072693,4.777232,40.760243,-74.190974
5,56,NaN,12.000000,14.993439,"POLYGON ((582452.999 700163.704, 582452.732 70...",6,1916.799532,268.558741,14.274713,134.279371,4.084662,40.755159,-74.173706
6,61,0.000000,16.000000,25.000001,"POLYGON ((579132.757 701443.286, 579132.994 70...",7,557.244583,118.985866,9.366568,59.492933,24.914234,40.758634,-74.185950
7,62,0.000000,16.000000,20.006562,"POLYGON ((579157.398 701432.869, 579157.162 70...",8,639.868920,107.992872,11.850207,53.996436,12.198195,40.758670,-74.185748
8,72,13.000000,16.000000,33.349739,"POLYGON ((578268.898 701786.897, 578268.637 70...",9,305.189798,101.669376,6.003574,50.834688,38.913959,40.759646,-74.188955
9,76,0.000000,14.000000,23.106956,"POLYGON ((581584.01 700481.631, 581583.776 700...",10,2043.087488,368.893146,11.076853,184.446573,0.162090,40.755965,-74.176772


In [90]:
GI_data_cut.to_excel(vector_dir/f'GI_points_{aoi}.xlsx')

In [91]:
GI_data_cut.to_file(vector_dir/f'GI_opportunities_{aoi}.shp')

In [38]:
gdf = gpd.read_file(vector_dir/'bloomfield_raw.shp')

In [39]:
gdf['geometry'] = gdf['geometry'].buffer(-2).buffer(2)

In [40]:
gdf.to_file(vector_dir/'GI_Opps_refined_geo.shp')

In [49]:
flow_paths = gpd.read_file(vector_dir/f'flow_{v_streams}.shp') 

In [55]:
flow_paths_buffer = flow_paths.copy()
flow_paths_buffer['geometry'] = flow_paths.buffer(distance=2)

In [56]:
gdf_split = gpd.overlay(gdf, flow_paths_buffer, how="difference", keep_geom_type=True)

In [58]:
gdf_split_exp = gdf_split.explode(ignore_index=True)

In [56]:
gdf_split_exp.to_file(vector_dir/f'{gi_raw.replace('raw','cesdfc')}.shp')

NameError: name 'gdf_split_exp' is not defined

In [30]:
#gs.run_command('r.stream.basins',direction =drain_dir,stream_rast =r_streams,basins=r_basins)


In [22]:
#gs.mapcalc("theshold_points = if({0} <= ({1} + 8) & !isnull({2}),1, null())".format(accum, threshold,r_streams))

In [ ]:
#gs.run_command('r.to.vect',input = r_basins,output = v_basins, type='area',column='basin_col', flags='s')

In [42]:
#gs.run_command('r.stream.order',elevation=demstream_rast=r_streams, direction=drain_dir, accumulation =accum,stream_vect =v_streams,strahler =r_streams_o)

In [43]:
#gs.run_command('r.stream.order',stream_rast=r_streams, direction=drain_dir,strahler =r_streams_o)

In [63]:
v_basins{aoi}

SyntaxError: unmatched '}' (292614801.py, line 1)

In [17]:
cell_size = (float(reg.nsres)+ float(reg.ewres)) /2

NameError: name 'reg' is not defined

## Watershed Delineation

In [22]:
#convert lines to points
from shapely.geometry import Point

gdf = gpd.read_file(vector_dir/'pipes.shp')
# Assuming 'gdf' is your existing GeoDataFrame with LineString geometries
gdf['geometry'] = gdf['geometry'].apply(lambda line: Point(line.coords[0]))

gdf.to_file(outlet_point)

/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Field INSTALLYEA created as String field, though DateTime requested.
  ogr_write(
/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Field LASTINSPEC created as String field, though DateTime requested.
  ogr_write(


In [23]:
#import outlet
gs.run_command('v.import', input= outlet_point,extent='region', output= v_outlet)

In [24]:
gs.run_command('r.stream.snap',input = v_outlet, output=f'{v_outlet}_snap',stream_rast = r_streams,radius = 3)

In [25]:
gs.run_command('v.db.connect', map=f'{v_outlet}_snap',table=v_outlet)

In [27]:
# query raster cells (a new column will be added to existing table)
gs.run_command('v.what.rast', map=f'{v_outlet}_snap', raster=accum, column='daa')

In [28]:
gs.run_command('v.out.ogr',input=f'{v_outlet}_snap',output=vector_dir/f'{v_outlet}_snap.geojson',type= 'point',format='GeoJSON')
#convert to shp
gpd.read_file(vector_dir/f'{v_outlet}_snap.geojson').to_file(vector_dir/f'{v_outlet}_snap.shp')

/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Field INSTALLYEA created as String field, though DateTime requested.
  ogr_write(
/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Field LASTINSPEC created as String field, though DateTime requested.
  ogr_write(


In [29]:
gs.run_command('r.stream.basins',points = f'{v_outlet}_snap', direction=drain_dir,basins=r_basins)

KeyboardInterrupt: 

In [73]:
gs.run_command('r.to.vect',input = r_basins,output = v_basins, type='area',column='basin_col', flags='s')

In [74]:
gs.run_command('v.out.ogr',input=v_basins,output=vector_dir/f'{v_basins}.geojson',type= 'area',format='GeoJSON')
#convert to shp
gpd.read_file(vector_dir/f'{v_basins}.geojson').to_file(vector_dir/f'{v_basins}.shp')

In [76]:
m = gj.InteractiveMap(height=600,width=1200)
m.add_vector(v_outlet)
m.add_vector(f'{v_outlet}_snap')
m.add_vector(v_basins)
m.show()

Map(center=[0.0, 0.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

## END